In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Mundka_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,NaN,210.0,NaN,195.0,276.0,321.0,166.0,52.0,161.0,214.0,346.0,330.0
1,2,340.0,255.0,144.0,217.0,265.0,209.0,237.0,88.0,70.0,268.0,313.0,307.0
2,3,374.0,220.0,140.0,244.0,331.0,188.0,235.0,71.0,128.0,249.0,423.0,310.0
3,4,394.0,328.0,136.0,239.0,370.0,269.0,97.0,55.0,118.0,241.0,398.0,216.0
4,5,347.0,149.0,126.0,234.0,360.0,314.0,151.0,38.0,75.0,204.0,412.0,221.0
5,6,321.0,160.0,172.0,242.0,325.0,211.0,172.0,51.0,143.0,190.0,419.0,254.0
6,7,366.0,215.0,246.0,NaN,378.0,270.0,99.0,44.0,84.0,173.0,430.0,270.0
7,8,370.0,210.0,198.0,296.0,257.0,288.0,87.0,NaN,199.0,222.0,398.0,369.0
8,9,365.0,183.0,177.0,299.0,204.0,190.0,209.0,NaN,266.0,216.0,373.0,247.0
9,10,309.0,333.0,241.0,264.0,169.0,176.0,260.0,96.0,172.0,174.0,357.0,279.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,339.617647,210.000000,186.235294,195.00000,276.000000,321.000000,166.0,52.000000,161.000000,214.0,346.000000,330.000000
1,2,340.000000,255.000000,144.000000,217.00000,265.000000,209.000000,237.0,88.000000,70.000000,268.0,313.000000,307.000000
2,3,374.000000,220.000000,140.000000,244.00000,331.000000,188.000000,235.0,71.000000,128.000000,249.0,423.000000,310.000000
3,4,394.000000,328.000000,136.000000,239.00000,370.000000,269.000000,97.0,55.000000,118.000000,241.0,398.000000,216.000000
4,5,347.000000,149.000000,126.000000,234.00000,360.000000,314.000000,151.0,38.000000,75.000000,204.0,412.000000,221.000000
5,6,321.000000,160.000000,172.000000,242.00000,325.000000,211.000000,172.0,51.000000,143.000000,190.0,419.000000,254.000000
6,7,366.000000,215.000000,246.000000,207.84375,244.057143,270.000000,99.0,44.000000,84.000000,173.0,430.000000,270.000000
7,8,370.000000,210.000000,198.000000,296.00000,257.000000,288.000000,87.0,63.235294,199.000000,222.0,398.000000,369.000000
8,9,365.000000,183.000000,177.000000,299.00000,204.000000,190.000000,209.0,63.235294,266.000000,216.0,373.000000,247.000000
9,10,309.000000,333.000000,241.000000,264.00000,169.000000,176.000000,143.2,96.000000,172.000000,174.0,357.000000,279.000000
